## Libraries

In [ ]:
# ------------------------------------------------------------------------------
# 1. Import Required Packages
# ------------------------------------------------------------------------------
import dask
import dask.dataframe as dd
from dask.distributed import Client
import hvplot.dask
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import pandas as pd
import pprint
import requests
import dask
import dask.dataframe as dd
from dask.distributed import Client
import numpy as np
import pandas as pd
import requests
import datetime
from io import StringIO
import earthaccess
import geopandas as gpd
import pandas as pd
from pathlib import Path
import glob
import numpy as np
from joblib import Parallel, delayed
import time
from shapely.geometry import LineString, Polygon
from shapely.ops import unary_union
from exactextract import exact_extract
import rasterio
import os
from pathlib import Path
from IPython.display import clear_output
import sys

## SWOT reach version D all observations with 11 filters

### Download SWOT reach version D

In [ ]:
client = Client(n_workers=48) 

In [ ]:
# Read the linkage file
usgs_link_merritt = pd.read_csv("/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv", dtype={'gage_id': object})

# Rename usgs_site_no to site_no
usgs_link_merritt = usgs_link_merritt.rename(columns={'gage_id': 'site_no'})

In [ ]:
def load_sword_nodes(sword_path, continents=None):
    """Load and merge SWORD nodes shapefiles from specified or all continents."""
    
    sword_dir = Path(sword_path)
    # all_continents = ['AF', 'AS', 'EU', 'NA', 'OC', 'SA']
    all_continents = ['NA']
    
    if continents is None:
        continents = all_continents
    
    nodes_files = []
    for continent in continents:
        pattern = str(sword_dir / continent / '*nodes*.shp')
        files = glob.glob(pattern)
        nodes_files.extend(files)
        print(f"{continent}: {len(files)} files found")
    
    if not nodes_files:
        raise FileNotFoundError(f"No nodes shapefiles found in {sword_path}")
    
    print(f"\nLoading {len(nodes_files)} total files...")
    
    nodes_list = [gpd.read_file(f) for f in sorted(nodes_files)]
    nodes = gpd.GeoDataFrame(
        pd.concat(nodes_list, ignore_index=True),
        crs=nodes_list[0].crs
    )
    
    # Free memory
    del nodes_list
    gc.collect()
    
    return nodes

In [ ]:
import pandas as pd
import geopandas as gpd
import requests
from io import StringIO
from pathlib import Path
import glob
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
import gc
import pyarrow.parquet as pq
import pyarrow as pa
import sys

SWORD_PATH = '/nas/cee-ice/data/SWORD/SWORD_v17b/shp'
nodes = load_sword_nodes(SWORD_PATH)

In [ ]:
nodes.columns

In [ ]:
usgs_link_merritt.columns

In [ ]:
def load_sword_reaches(sword_path, continents=None):
    """Load and merge SWORD reaches shapefiles from specified or all continents."""
    
    sword_dir = Path(sword_path)
    
    all_continents = ['AF', 'AS', 'EU', 'NA', 'OC', 'SA']
    
    if continents is None:
        continents = all_continents
    
    reaches_files = []
    for continent in continents:
        pattern = str(sword_dir / continent / '*reaches*.shp')
        files = glob.glob(pattern)
        reaches_files.extend(files)
        print(f"{continent}: {len(files)} files found")
    
    if not reaches_files:
        raise FileNotFoundError(f"No reaches shapefiles found in {sword_path}")
    
    print(f"\nLoading {len(reaches_files)} total files...")
    
    # Read and merge all reaches
    reaches_list = [gpd.read_file(f) for f in sorted(reaches_files)]
    reaches = gpd.GeoDataFrame(
        pd.concat(reaches_list, ignore_index=True),
        crs=reaches_list[0].crs
    )
    
    return reaches

sword_path = '/nas/cee-ice/data/SWORD/SWORD_v17b/shp'
reaches = load_sword_reaches(sword_path)
# reaches = load_sword_reaches(sword_path, continents=['NA', 'SA'])

In [ ]:
len(node_ids)

In [ ]:
#### Configuration ####
# Your reach IDs
# reach_ids = usgs_link_merritt.reach_id.drop_duplicates().to_list()
reach_ids = reaches.reach_id.drop_duplicates().to_list()

# Hydrocron API URL
HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"

# Collection name (Version D data)
COLLECTION_NAME = "SWOT_L2_HR_RiverSP_D"  

# Time range
start_time = "2023-07-27T00:00:00Z"
end_time = "2026-01-31T23:59:59Z"

# Fields to download
fields = (
    "reach_id,time_str,"
    "wse,wse_u,width,slope2,"
    "reach_q,partial_f,ice_clim_f,xovr_cal_q,reach_q_b,dark_frac,xtrk_dist,"
    "area_detct,area_total,"
    "p_width,p_length,p_wse,"
    "ice_dyn_f,obs_frac_n,p_lat,p_lon,crid"
)

In [ ]:
# SWOT L2_HR_RiverSP_D Version D - All 127 fields (2026/01)
all_riversp_fields = (
    # Identifiers (4)
    "reach_id,time,time_tai,time_str,"
    
    # Location (3)
    "p_lat,p_lon,river_name,"
    
    # Water Surface Elevation (5)
    "wse,wse_u,wse_r_u,wse_c,wse_c_u,"
    
    # Slope (6)
    "slope,slope_u,slope_r_u,slope2,slope2_u,slope2_r_u,"
    
    # Width (4)
    "width,width_u,width_c,width_c_u,"
    
    # Area (5)
    "area_total,area_tot_u,area_detct,area_det_u,area_wse,"
    
    # Cross-section area change (2)
    "d_x_area,d_x_area_u,"
    
    # Geometry/Distance (4)
    "layovr_val,node_dist,loc_offset,xtrk_dist,"
    
    # Discharge - Consensus (4)
    "dschg_c,dschg_c_u,dschg_csf,dschg_c_q,"
    # Discharge - Gauge-constrained Consensus (4)
    "dschg_gc,dschg_gc_u,dschg_gcsf,dschg_gc_q,"
    # Discharge - MetroMan (4)
    "dschg_m,dschg_m_u,dschg_msf,dschg_m_q,"
    # Discharge - Gauge-constrained MetroMan (4)
    "dschg_gm,dschg_gm_u,dschg_gmsf,dschg_gm_q,"
    # Discharge - BAM (4)
    "dschg_b,dschg_b_u,dschg_bsf,dschg_b_q,"
    # Discharge - Gauge-constrained BAM (4)
    "dschg_gb,dschg_gb_u,dschg_gbsf,dschg_gb_q,"
    # Discharge - HiVDI (4)
    "dschg_h,dschg_h_u,dschg_hsf,dschg_h_q,"
    # Discharge - Gauge-constrained HiVDI (4)
    "dschg_gh,dschg_gh_u,dschg_ghsf,dschg_gh_q,"
    # Discharge - MOMMA (4)
    "dschg_o,dschg_o_u,dschg_osf,dschg_o_q,"
    # Discharge - Gauge-constrained MOMMA (4)
    "dschg_go,dschg_go_u,dschg_gosf,dschg_go_q,"
    # Discharge - SAD (4)
    "dschg_s,dschg_s_u,dschg_ssf,dschg_s_q,"
    # Discharge - Gauge-constrained SAD (4)
    "dschg_gs,dschg_gs_u,dschg_gssf,dschg_gs_q,"
    # Discharge - SIC4DVar (4)
    "dschg_i,dschg_i_u,dschg_isf,dschg_i_q,"
    # Discharge - Gauge-constrained SIC4DVar (4)
    "dschg_gi,dschg_gi_u,dschg_gisf,dschg_gi_q,"
    # Discharge quality flags (4)
    "dschg_q_b,dschg_gq_b,reach_q,reach_q_b,"
    
    # Quality flags (7)
    "dark_frac,ice_clim_f,ice_dyn_f,partial_f,n_good_nod,obs_frac_n,xovr_cal_q,"
    
    # Geophysical corrections (10)
    "geoid_hght,geoid_slop,solid_tide,load_tidef,load_tideg,pole_tide,"
    "dry_trop_c,wet_trop_c,iono_c,xovr_cal_c,"
    
    # Reach connectivity (4)
    "n_reach_up,n_reach_dn,rch_id_up,rch_id_dn,"
    
    # Prior database info (12)
    "p_wse,p_wse_var,p_width,p_wid_var,p_n_nodes,p_dist_out,"
    "p_length,p_maf,p_dam_id,p_n_ch_max,p_n_ch_mod,p_low_slp,"
    
    # Geometry (1)
    "geometry"
)

all_test_fields = [f.strip() for f in all_riversp_fields.split(',') if f.strip()]

In [ ]:
import pandas as pd
import requests
from io import StringIO
import time
import numpy as np

# Configuration
HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"
COLLECTION_NAME = "SWOT_L2_HR_RiverSP_D"
start_time = "2023-07-27T00:00:00Z"
end_time = "2026-01-31T23:59:59Z"

fields = all_test_fields

test_reach_ids = reach_ids[:100]

def benchmark_single_request(reach_id):
    """단일 요청의 각 단계별 시간 측정"""
    
    params = {
        "feature": "Reach",
        "feature_id": reach_id,
        "output": "csv",
        "start_time": start_time,
        "end_time": end_time,
        "fields": fields,
        "collection_name": COLLECTION_NAME
    }
    
    # 1. HTTP 요청/응답 시간
    t0 = time.perf_counter()
    response = requests.get(HYDROCRON_URL, params=params, timeout=60)
    t1 = time.perf_counter()
    http_time = t1 - t0
    
    # 2. JSON 파싱 시간
    t2 = time.perf_counter()
    data = response.json()
    t3 = time.perf_counter()
    json_time = t3 - t2
    
    # 3. CSV -> DataFrame 변환 시간
    csv_time = 0
    rows = 0
    if "results" in data and data["results"].get("csv"):
        csv_str = data["results"]["csv"]
        t4 = time.perf_counter()
        df = pd.read_csv(StringIO(csv_str))
        t5 = time.perf_counter()
        csv_time = t5 - t4
        rows = len(df)
    
    return {
        "reach_id": reach_id,
        "http_time": http_time,
        "json_time": json_time,
        "csv_time": csv_time,
        "total_time": http_time + json_time + csv_time,
        "rows": rows
    }


if __name__ == "__main__":
    
    print("=" * 60)
    print("Hydrocron API Benchmark (100 reaches)")
    print("=" * 60)
    
    results = []
    
    for i, reach_id in enumerate(test_reach_ids[:100]):
        try:
            result = benchmark_single_request(reach_id)
            results.append(result)
            
            if (i + 1) % 10 == 0:
                print(f"Progress: {i+1}/100")
        except Exception as e:
            print(f"Error on {reach_id}: {e}")
    
    # 결과 분석
    df = pd.DataFrame(results)
    
    print("\n" + "=" * 60)
    print("Results Summary")
    print("=" * 60)
    
    print(f"\nSample size: {len(df)}")
    print(f"Total rows downloaded: {df['rows'].sum()}")
    
    print("\n--- Time per request (seconds) ---")
    print(f"HTTP 요청/응답:  mean={df['http_time'].mean():.3f}s, std={df['http_time'].std():.3f}s")
    print(f"JSON 파싱:       mean={df['json_time'].mean():.4f}s, std={df['json_time'].std():.4f}s")
    print(f"CSV→DataFrame:  mean={df['csv_time'].mean():.4f}s, std={df['csv_time'].std():.4f}s")
    print(f"Total:          mean={df['total_time'].mean():.3f}s")
    
    print("\n--- Percentage breakdown ---")
    total = df['http_time'].sum() + df['json_time'].sum() + df['csv_time'].sum()
    print(f"HTTP 요청/응답:  {df['http_time'].sum() / total * 100:.1f}%")
    print(f"JSON 파싱:       {df['json_time'].sum() / total * 100:.1f}%")
    print(f"CSV→DataFrame:  {df['csv_time'].sum() / total * 100:.1f}%")
    
    print("\n--- Estimated full download time ---")
    total_reaches = 500000  # 대략적인 전체 reach 수
    
    # Sequential
    seq_time = df['total_time'].mean() * total_reaches
    print(f"Sequential:           {seq_time/3600:.1f} hours")
    
    # ThreadPool 예상
    for workers in [100, 200, 300]:
        parallel_time = (df['total_time'].mean() * total_reaches) / workers
        print(f"ThreadPool({workers}):     {parallel_time/3600:.1f} hours")

In [ ]:
import requests

sample_reach = reach_ids[0]
HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"
COLLECTION_NAME = "SWOT_L2_HR_RiverSP_D"
start_time = "2025-12-01T00:00:00Z"
end_time = "2026-01-31T23:59:59Z"

valid_fields = []
invalid_fields = []

for field in all_test_fields:
    params = {
        "feature": "Reach",
        "feature_id": sample_reach,
        "output": "csv",
        "start_time": start_time,
        "end_time": end_time,
        "fields": f"reach_id,{field}",  # reach_id는 항상 포함
        "collection_name": COLLECTION_NAME
    }
    
    response = requests.get(HYDROCRON_URL, params=params)
    
    if response.status_code == 200 and "results" in response.json():
        valid_fields.append(field)
        print(f"✓ {field}")
    else:
        invalid_fields.append(field)
        print(f"✗ {field}")

print("\n" + "="*50)
print(f"Valid fields ({len(valid_fields)}):")
print(",".join(valid_fields))
print(f"\nInvalid fields ({len(invalid_fields)}):")
print(",".join(invalid_fields))

Group: reach
Variables: dict_keys(['reach_id', 'time', 'time_str', 'd_x_area', 'd_x_area_u', 'slope', 'slope_u', 'slope_r_u', 'slope2', 'slope2_u', 'slope2_r_u', 'width', 'p_width', 'width_u', 'wse', 'wse_u', 'wse_r_u', 'p_length', 'reach_q', 'reach_q_b', 'dark_frac', 'ice_clim_f', 'ice_dyn_f', 'partial_f', 'n_good_nod', 'obs_frac_n', 'xovr_cal_q', 'xtrk_dist'])

Group: node
Variables: dict_keys(['reach_id', 'node_id', 'time', 'time_str', 'd_x_area', 'd_x_area_u', 'slope', 'slope_u', 'slope_r_u', 'slope2', 'slope2_u', 'slope2_r_u', 'width', 'p_width', 'width_u', 'wse', 'wse_u', 'wse_r_u', 'node_q', 'node_q_b', 'dark_frac', 'ice_clim_f', 'ice_dyn_f', 'n_good_pix', 'xovr_cal_q', 'xtrk_dist'])

In [ ]:
@dask.delayed
def query_hydrocron(query_url, reach_id, start_time, end_time, fields, collection_name, empty_df):  # add collection_name
    """Query Hydrocron for reach-level time series data."""
    
    params = {
        "feature": "Reach",
        "feature_id": reach_id,
        "output": "csv",
        "start_time": start_time,
        "end_time": end_time,
        "fields": fields,
        "collection_name": collection_name  # add!!!
    }
    
    results = requests.get(query_url, params=params)
    
    if "results" in results.json().keys():
        results_csv = results.json()["results"]["csv"]
        df = pd.read_csv(StringIO(results_csv))
    else:
        df = empty_df
    
    return df

#### Main Execution ####
if __name__ == "__main__":
    
    print("Starting SWOT Data Download")
    print(f"Number of reaches: {len(reach_ids)}")
    print(f"Time range: {start_time} to {end_time}")
    print(f"Fields: {fields}")
    print()
    
    # Create Dask client
    print("Creating Dask client...")
    client = Client(n_workers=48)  # Adjust n_workers based on your CPU cores
    print(f"Dask dashboard: {client.dashboard_link}")
    print()
    
    try:
        # Get metadata from first successful query
        print("Getting metadata schema...")
        meta_df = None
        for sample_reach in reach_ids[:10]:  
            sample_params = {
                "feature": "Reach",
                "feature_id": sample_reach,
                "output": "csv",
                "start_time": start_time,
                "end_time": end_time,
                "fields": fields,
                "collection_name": COLLECTION_NAME
            }
            
            try:
                sample_result = requests.get(HYDROCRON_URL, params=sample_params, timeout=30)
                if sample_result.status_code == 200 and sample_result.text:
                    sample_json = sample_result.json()
                    if "results" in sample_json.keys() and sample_json["results"]["csv"]:
                        meta_df = pd.read_csv(StringIO(sample_json["results"]["csv"])).head(0)
                        print(f"Got metadata from reach {sample_reach}")
                        print(f"Columns: {list(meta_df.columns)}")
                        break
            except:
                continue
        
        if meta_df is None:
            raise ValueError("Could not get metadata schema from any reach")
        
        # Create queries that return Pandas.DataFrame objects
        print("Creating delayed queries...")
        results = []
        
        for reach in reach_ids:
            empty_df = meta_df.copy()  
            
            results.append(
                query_hydrocron(
                    HYDROCRON_URL, 
                    reach,  # modified (reach_ids -> reach)
                    start_time, 
                    end_time, 
                    all_test_fields, 
                    COLLECTION_NAME,
                    empty_df
                )
            )
        
        print(f"Created {len(results)} queries")
        print()
        
        # Load DataFrame results into dask.dataframe
        print("Loading into Dask DataFrame and computing...")
        ddf = dd.from_delayed(results, meta=meta_df) 
        
        # Compute to get actual data
        df = ddf.compute()
        
        print(f"Downloaded {len(df)} total rows")
        print()
        
        # Remove fill values for missing observations
        print("Cleaning data...")
        df = df.loc[(df["wse"] != -999999999999.0)]
        
        # Convert time_str to datetime format
        df['time_str'] = pd.to_datetime(df['time_str'])
        
        print(f"After cleaning: {len(df)} rows")
        print()
        
        # Save to CSV
        output_file = "swot_data_selected_reaches.parquet"
        df.to_parquet(output_file)#, index=False)
        print(f"Data saved to: {output_file}")
        print()
        
        # Display summary
        print("="*70)
        print("Summary")
        print("="*70)
        print(f"Total observations: {len(df)}")
        print(f"Number of reaches with data: {df['reach_id'].nunique()}")
        print()
        print("Observations per reach:")
        print(df.groupby('reach_id').size().sort_values(ascending=False))
        print()
        
        if 'time_str' in df.columns:
            print(f"Time range:")
            print(f"  Start: {df['time_str'].min()}")
            print(f"  End: {df['time_str'].max()}")
        print()
        
        print("Columns:")
        print(df.columns.tolist())
        print()
        
        print("First few rows:")
        print(df.head(10))
        
    finally:
        # Close client
        print()
        print("Closing Dask client...")
        client.close()
        print("Done!")

In [ ]:
df.crid.unique()

In [ ]:
from datetime import timedelta
def find_missing_periods(df, date_col, start_date, end_date):
    existing = set(pd.to_datetime(df[date_col]).dt.date)
    all_dates = set(pd.date_range(start_date, end_date, freq='D').date)
    missing = sorted(all_dates - existing)
    
    if not missing:
        return []
    
    periods = []
    start = end = missing[0]
    
    for d in missing[1:]:
        if d - end == timedelta(days=1):
            end = d
        else:
            periods.append((start, end))
            start = end = d
    periods.append((start, end))
    return periods


def print_periods(periods):
    for s, e in periods:
        days = (e - s).days + 1
        print(f"{s} ~ {e} ({days}d)" if s != e else f"{s} (1d)")

periods = find_missing_periods(df, 'time_str', start_time, end_time)
print_periods(periods)

In [ ]:
df.crid.unique()

### Filtering out (11 filters)

In [ ]:
df_swot = df.copy()

def apply_harlan_filters(df_swot, waterbody_type='river', hampel_k=2, hampel_gap_days=5, verbose=True):
    """
    Apply 11 quality filters following Harlan et al. (2026)
    
    Inputs
    ------
    df_swot : SWOT data from Hydrocron API
    waterbody_type : 'river' or 'lake' - affects size threshold
    hampel_k : Half-window size for Hampel filter
    hampel_gap_days : Maximum time gap (in days) between consecutive observations

    
    Outputs
    --------
    filtered_df : Filtered observations
    filter_stats : Statistics for each filter
    """
    
    hampel_window_size = 2 * hampel_k + 1
    
    if verbose:
        print(f"Original data: {len(df_swot):,} observations")
        print(f"Waterbody type: {waterbody_type}")
        print(f"Hampel filter settings: K={hampel_k} (window={hampel_window_size} points), gap threshold={hampel_gap_days} days")
    
    # Initialize
    df = df_swot.copy()
    filter_stats = {}
    
    # Store Hampel parameters
    filter_stats['hampel_parameters'] = {
        'k': hampel_k,
        'window_size': hampel_window_size,
        'gap_threshold_days': hampel_gap_days,
        'min_sequence_length': hampel_window_size
    }
    
    #### PHASE 1: Design Limitation Filters ####   
    if verbose:
        print("PHASE 1: DESIGN LIMITATION FILTERS")
    
    # Filter 1: Ice Climatology Flag < 1
    mask_ice = df['ice_clim_f'] < 1
    n_ice_pass = mask_ice.sum()
    n_ice_fail = (~mask_ice).sum()
    filter_stats['ice_climatology'] = {
        'pass': n_ice_pass,
        'fail': n_ice_fail,
        'pass_rate': n_ice_pass / len(df) * 100
    }
    if verbose:
        print(f"1. Ice Climatology (<1):")
        print(f"   Pass: {n_ice_pass:,} | Fail: {n_ice_fail:,} | Rate: {filter_stats['ice_climatology']['pass_rate']:.1f}%")
    
    # Filter 2: Cross Track Distance 10-60 km
    mask_xtrk = (df['xtrk_dist'].abs() >= 10000) & (df['xtrk_dist'].abs() <= 60000)
    n_xtrk_pass = mask_xtrk.sum()
    n_xtrk_fail = (~mask_xtrk).sum()
    filter_stats['cross_track'] = {
        'pass': n_xtrk_pass,
        'fail': n_xtrk_fail,
        'pass_rate': n_xtrk_pass / len(df) * 100
    }
    if verbose:
        print(f"2. Cross Track Distance (10-60 km):")
        print(f"   Pass: {n_xtrk_pass:,} | Fail: {n_xtrk_fail:,} | Rate: {filter_stats['cross_track']['pass_rate']:.1f}%")
    
    # Filter 3: Waterbody Size Thresholds
    if waterbody_type == 'river':
        mask_size = (df['p_width'] >= 80) & (df['p_length'] >= 7000)
        size_desc = "River (width≥80m, length≥7km)"
    else:
        df['area_km2'] = df['area_total'] / 1e6
        mask_size = df['area_km2'] >= 0.0625
        size_desc = "Lake (area≥0.0625 km²)"
    
    n_size_pass = mask_size.sum()
    n_size_fail = (~mask_size).sum()
    filter_stats['waterbody_size'] = {
        'pass': n_size_pass,
        'fail': n_size_fail,
        'pass_rate': n_size_pass / len(df) * 100
    }
    if verbose:
        print(f"3. Waterbody Size ({size_desc}):")
        print(f"   Pass: {n_size_pass:,} | Fail: {n_size_fail:,} | Rate: {filter_stats['waterbody_size']['pass_rate']:.1f}%")
    
    # Apply all design limitation filters
    mask_design = mask_ice & mask_xtrk & mask_size
    df_design = df[mask_design].copy()
    
    adjusted_denominator = len(df_design)
    n_design_removed = len(df) - adjusted_denominator
    
    if verbose:
        print(f"  Design-filtered dataset: {adjusted_denominator:,} observations")
        print(f"  Removed: {n_design_removed:,} ({n_design_removed/len(df)*100:.1f}%)")
        print(f"  This is the ADJUSTED DENOMINATOR (TRUE 100%)")
    
    if adjusted_denominator == 0:
        if verbose:
            print("ARNING: No observations passed design filters!")
        return df_design, filter_stats
    
    #### PHASE 2: Quality Filters ####   
    if verbose:
        print("PHASE 2: QUALITY FILTERS (on design-filtered data)")
    
    df_quality = df_design.copy()
    
    # Filter 4: Summary Quality Flag < 4
    mask_summary = df_quality['reach_q'] < 4
    n_summary_pass = mask_summary.sum()
    filter_stats['summary_quality'] = {
        'pass': n_summary_pass,
        'pass_rate': n_summary_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"4. Summary Quality (<4):")
        print(f"   Pass: {n_summary_pass:,} / {adjusted_denominator:,} ({filter_stats['summary_quality']['pass_rate']:.1f}%)")
    
    # Filter 5: Partial Coverage Flag = 0
    mask_partial = df_quality['partial_f'] == 0
    n_partial_pass = mask_partial.sum()
    filter_stats['partial_coverage'] = {
        'pass': n_partial_pass,
        'pass_rate': n_partial_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"5. Partial Coverage (=0):")
        print(f"   Pass: {n_partial_pass:,} / {adjusted_denominator:,} ({filter_stats['partial_coverage']['pass_rate']:.1f}%)")
    
    # Filter 6: Crossover Calibration Quality < 2
    if 'xovr_cal_q' in df_quality.columns:
        mask_xovr = df_quality['xovr_cal_q'] < 2
        n_xovr_pass = mask_xovr.sum()
        filter_stats['crossover_cal'] = {
            'pass': n_xovr_pass,
            'pass_rate': n_xovr_pass / adjusted_denominator * 100
        }
        if verbose:
            print(f"6. Crossover Calibration (<2):")
            print(f"   Pass: {n_xovr_pass:,} / {adjusted_denominator:,} ({filter_stats['crossover_cal']['pass_rate']:.1f}%)")
    else:
        mask_xovr = pd.Series([True] * len(df_quality), index=df_quality.index)
        filter_stats['crossover_cal'] = {'pass': len(df_quality), 'pass_rate': 100.0}
        if verbose:
            print("6. Crossover Calibration: ⚠️  Field not available (skipping)")
    
    # Filter 7: Bitwise Quality Flag < 4,194,304
    mask_bitwise = df_quality['reach_q_b'] < 4194304
    n_bitwise_pass = mask_bitwise.sum()
    filter_stats['bitwise_quality'] = {
        'pass': n_bitwise_pass,
        'pass_rate': n_bitwise_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"7. Bitwise Quality (<4,194,304):")
        print(f"   Pass: {n_bitwise_pass:,} / {adjusted_denominator:,} ({filter_stats['bitwise_quality']['pass_rate']:.1f}%)")
    
    # Filter 8: Dark Water Fraction ≤ 0.5
    mask_dark = df_quality['dark_frac'] <= 0.5
    n_dark_pass = mask_dark.sum()
    filter_stats['dark_water'] = {
        'pass': n_dark_pass,
        'pass_rate': n_dark_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"8. Dark Water Fraction (≤0.5):")
        print(f"   Pass: {n_dark_pass:,} / {adjusted_denominator:,} ({filter_stats['dark_water']['pass_rate']:.1f}%)")
    
    # Filter 9: Observed Area Fraction >= 0.5
    df_quality['obs_area_frac'] = df_quality['area_detct'] / df_quality['area_total']
    mask_area = df_quality['obs_area_frac'] >= 0.5
    n_area_pass = mask_area.sum()
    filter_stats['observed_area'] = {
        'pass': n_area_pass,
        'pass_rate': n_area_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"9. Observed Area Fraction (>=0.5):")
        print(f"   Pass: {n_area_pass:,} / {adjusted_denominator:,} ({filter_stats['observed_area']['pass_rate']:.1f}%)")
    
    # Filter 10: WSE Prior Difference < 50m
    df_quality['wse_prior_diff'] = (df_quality['wse'] - df_quality['p_wse']).abs()
    mask_prior = df_quality['wse_prior_diff'] < 50
    n_prior_pass = mask_prior.sum()
    filter_stats['wse_prior'] = {
        'pass': n_prior_pass,
        'pass_rate': n_prior_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"10. WSE Prior Difference (<50m):")
        print(f"   Pass: {n_prior_pass:,} / {adjusted_denominator:,} ({filter_stats['wse_prior']['pass_rate']:.1f}%)")
    
    # Combine quality filters (excluding Hampel)
    mask_quality_pre_hampel = (
        mask_summary & mask_partial & mask_xovr & mask_bitwise & 
        mask_dark & mask_area & mask_prior
    )
    df_pre_hampel = df_quality[mask_quality_pre_hampel].copy()
    n_pre_hampel = len(df_pre_hampel)
    
    if verbose:
        print()
        print(f"After filters 4-10: {n_pre_hampel:,} / {adjusted_denominator:,} ({n_pre_hampel/adjusted_denominator*100:.1f}%)")
        print()
    
    # ============================================================
    # Filter 11: Hampel Outlier Filter
    # ============================================================
    
    if verbose:
        print(f"   11. Hampel Outlier Filter:")
        print(f"   Settings: h=3, K={hampel_k}, window={hampel_window_size} points, gap threshold={hampel_gap_days} days")
        print(f"   Calculating per-reach outliers with temporal gap detection...")
    
    df_pre_hampel['hampel_outlier'] = 0
    hampel_stats = {
        'total_checked': 0, 
        'outliers_found': 0, 
        'reaches_processed': 0,
        'sequences_found': 0,
        'sequences_too_short': 0
    }
    
    # Apply Hampel filter per reach
    for reach_id in df_pre_hampel['reach_id'].unique():
        reach_mask = df_pre_hampel['reach_id'] == reach_id
        reach_data = df_pre_hampel[reach_mask].sort_values('time_str').copy()
        
        if len(reach_data) < hampel_window_size:
            continue
        
        hampel_stats['reaches_processed'] += 1
        sequences = split_into_temporal_sequences(reach_data, gap_threshold_days=hampel_gap_days)
        hampel_stats['sequences_found'] += len(sequences)
        
        for seq_indices in sequences:  # ← 들여쓰기 수정
            seq_data = reach_data.loc[seq_indices]
            wse_series = seq_data['wse'].values
            
            outlier_indices = hampel_filter_harlan(wse_series, h=3, k=hampel_k)
            
            if len(outlier_indices) > 0:
                original_indices = seq_indices[outlier_indices]
                df_pre_hampel.loc[original_indices, 'hampel_outlier'] = 1
                hampel_stats['outliers_found'] += len(outlier_indices)
            
            hampel_stats['total_checked'] += len(seq_indices)

    mask_hampel = df_pre_hampel['hampel_outlier'] == 0
    n_hampel_pass = mask_hampel.sum()
    n_hampel_fail = (~mask_hampel).sum()

    filter_stats['hampel_outlier'] = {
        'pass': n_hampel_pass,
        'fail': n_hampel_fail,
        'pass_rate': n_hampel_pass / adjusted_denominator * 100,
        'reaches_processed': hampel_stats['reaches_processed'],
        'sequences_found': hampel_stats['sequences_found'],
        'sequences_too_short': hampel_stats['sequences_too_short']
    }
    
    if verbose:
        print(f"   Reaches processed: {hampel_stats['reaches_processed']}")
        print(f"   Temporal sequences found: {hampel_stats['sequences_found']}")
        print(f"   Sequences too short (<{hampel_window_size} obs): {hampel_stats['sequences_too_short']}")
        print(f"   Outliers detected: {n_hampel_fail:,} / {hampel_stats['total_checked']:,} ({n_hampel_fail/max(1,hampel_stats['total_checked'])*100:.1f}%)")
        print(f"   Pass: {n_hampel_pass:,} / {adjusted_denominator:,} ({filter_stats['hampel_outlier']['pass_rate']:.1f}%)")
        print(f"   Note: {hampel_window_size}-point window (K={hampel_k}), sequences split at >{hampel_gap_days} day gaps")
    
    # Final filtered dataset
    filtered_df = df_pre_hampel[mask_hampel].copy()
    final_count = len(filtered_df)
    
    # Summary statistics
    if verbose:
        print()
        print(f"FILTERING SUMMARY")
        print(f"{'='*70}")
        print(f"Original observations:              {len(df):>10,}")
        print(f"Design limitation filters removed:  {n_design_removed:>10,} ({n_design_removed/len(df)*100:>5.1f}%)")
        print(f"Adjusted denominator (TRUE 100%):   {adjusted_denominator:>10,}")
        print(f"Quality filters removed:            {adjusted_denominator - final_count:>10,} ({(adjusted_denominator - final_count)/adjusted_denominator*100:>5.1f}%)")
        print(f"Final observations:                 {final_count:>10,}")
        print(f"\nOverall pass-through rate:")
        print(f"  - vs. adjusted denominator:       {final_count/adjusted_denominator*100:>10.1f}%")
        print(f"  - vs. original data:              {final_count/len(df)*100:>10.1f}%")
        print(f"\nHampel filter configuration:")
        print(f"  - K={hampel_k}, Window={hampel_window_size} points, Gap threshold={hampel_gap_days} days")
        print()
    
    filter_stats['summary'] = {
        'original_count': len(df),
        'design_removed': n_design_removed,
        'adjusted_denominator': adjusted_denominator,
        'quality_removed': adjusted_denominator - final_count,
        'final_count': final_count,
        'pass_rate_adjusted': final_count / adjusted_denominator * 100 if adjusted_denominator > 0 else 0,
        'pass_rate_original': final_count / len(df) * 100
    }
    
    return filtered_df, filter_stats


def split_into_temporal_sequences(reach_data, gap_threshold_days=5):
    """Split reach data into continuous temporal sequences based on time gaps"""
    import pandas as pd
    
    if not pd.api.types.is_datetime64_any_dtype(reach_data['time_str']):
        reach_data = reach_data.copy()
        reach_data['time_str'] = pd.to_datetime(reach_data['time_str'])
    
    times = reach_data['time_str'].values
    indices = reach_data.index.values
    
    sequences = []
    current_sequence = [indices[0]]
    
    for i in range(1, len(indices)):
        time_diff = (times[i] - times[i-1]) / np.timedelta64(1, 'D')
        
        if time_diff <= gap_threshold_days:
            current_sequence.append(indices[i])
        else:
            sequences.append(np.array(current_sequence))
            current_sequence = [indices[i]]
    
    if len(current_sequence) > 0:
        sequences.append(np.array(current_sequence))
    
    return sequences


def hampel_filter_harlan(series, h=3, k=2):
    """Hampel outlier filter with one-sided windows at edges"""
    import numpy as np
    
    window_size = 2 * k + 1
    
    # if len(series) < window_size:
    #     return []
    
    series = np.array(series)
    n = len(series)
    outliers = []
    
    # Process ALL points including edges
    for i in range(n):
        # Determine window boundaries
        if i < k:
            # Left edge: extend window to right
            window_start = 0
            window_end = min(window_size, n)
        elif i >= n - k:
            # Right edge: extend window to left
            window_start = max(0, n - window_size)
            window_end = n
        else:
            # Center: symmetric window
            window_start = i - k
            window_end = i + k + 1
        
        window = series[window_start:window_end]
        median_i = np.median(window)
        mad_i = np.median(np.abs(window - median_i))
        
        if mad_i == 0:
            if series[i] != median_i:
                outliers.append(i)
            continue
        
        scaled_mad = 1.4826 * mad_i
        deviation = abs(series[i] - median_i)
        
        if deviation > h * scaled_mad:
            outliers.append(i)
    
    return outliers


def test_hampel_settings(df, k_values=[1, 2, 3, 5, 10], gap_days=[3, 5, 7, 10]):
    """
    Test multiple combinations of K and gap_days - ONLY PRINTS COMPARISON TABLE
    
    Input
    -----
    df : SWOT data
    k_values : K values to test
    gap_days : Gap threshold values to test (in days)
    """
    results = []
    
    for k in k_values:
        for gap in gap_days:
            # Run filtering with verbose=False (no printing)
            df_filtered, stats = apply_harlan_filters(
                df, 
                waterbody_type='river',
                hampel_k=k,
                hampel_gap_days=gap,
                verbose=False  # ← SUPPRESS ALL OUTPUT
            )
            
            results.append({
                'k': k,
                'window_size': 2*k+1,
                'gap_days': gap,
                'final_count': stats['summary']['final_count'],
                'pass_rate': stats['summary']['pass_rate_adjusted'],
                'outliers': stats['hampel_outlier']['fail'],
                'sequences': stats['hampel_outlier']['sequences_found']
            })
    
    # ONLY print comparison table
    print("="*84)
    print("COMPARISON SUMMARY")
    print("="*84)
    print(f"{'K':<4} {'Window':<8} {'Gap(d)':<8} {'Final':<10} {'Pass%':<8} {'Outliers':<10} {'Sequences':<10}")
    print("─"*84)
    
    for r in results:
        print(f"{r['k']:<4} {r['window_size']:<8} {r['gap_days']:<8} "
              f"{r['final_count']:<10,} {r['pass_rate']:<8.1f} "
              f"{r['outliers']:<10,} {r['sequences']:<10}")
    
    return pd.DataFrame(results)

# Only comparison table will be printed
comparison_df = test_hampel_settings(
    df_swot, 
    k_values=[1, 2, 5, 7], 
    gap_days=[1, 3, 5]
)

In [ ]:
df_filtered, stats = apply_harlan_filters(
    df_swot, 
    hampel_k=7, 
    hampel_gap_days=3
)

df_filtered.to_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/swotall_qc_filtered_vD_ver2.parquet')

## SWOT node version D all observations with 11 filters

### Download SWOT node version D

In [ ]:
client = Client(n_workers=48) 

In [ ]:
def load_sword_reaches(sword_path, continents=None):
    """Load and merge SWORD reaches shapefiles from specified or all continents."""
    
    sword_dir = Path(sword_path)
    
    # all_continents = ['AF', 'AS', 'EU', 'NA', 'OC', 'SA']
    all_continents = ['NA']
    
    if continents is None:
        continents = all_continents
    
    reaches_files = []
    for continent in continents:
        pattern = str(sword_dir / continent / '*nodes*.shp')
        files = glob.glob(pattern)
        reaches_files.extend(files)
        print(f"{continent}: {len(files)} files found")
    
    if not reaches_files:
        raise FileNotFoundError(f"No reaches shapefiles found in {sword_path}")
    
    print(f"\nLoading {len(reaches_files)} total files...")
    
    # Read and merge all reaches
    reaches_list = [gpd.read_file(f) for f in sorted(reaches_files)]
    reaches = gpd.GeoDataFrame(
        pd.concat(reaches_list, ignore_index=True),
        crs=reaches_list[0].crs
    )
    
    return reaches

sword_path = '/nas/cee-ice/data/SWORD/SWORD_v17b/shp'
nodes = load_sword_reaches(sword_path)
# reaches = load_sword_reaches(sword_path, continents=['NA', 'SA'])

In [ ]:
# 방법 2: Earthdata에서 다운로드 (earthaccess 라이브러리 사용)
import earthaccess
import geopandas as gpd
import zipfile
import os

# Earthdata 로그인
earthaccess.login()

# Version D reach 파일 검색 (최근 1개만)
results = earthaccess.search_data(
    short_name="SWOT_L2_HR_RiverSP_node_D",
    temporal=("2026-01-01", "2026-02-17"),
    count=1
)

# 다운로드
downloaded = earthaccess.download(results, "./swot_temp")
print(f"Downloaded: {downloaded}")

# Shapefile 읽기
for f in downloaded:
    if str(f).endswith('.zip'):
        with zipfile.ZipFile(f, 'r') as z:
            z.extractall("./swot_temp/extracted")
        
        # .shp 파일 찾기
        for root, dirs, files in os.walk("./swot_temp/extracted"):
            for file in files:
                if file.endswith('.shp') and 'Node' in file:
                    shp_path = os.path.join(root, file)
                    gdf = gpd.read_file(shp_path)
                    
                    print("\n" + "="*60)
                    print(f"File: {file}")
                    print("="*60)
                    print(f"\nAll fields ({len(gdf.columns)}):")
                    for i, col in enumerate(gdf.columns):
                        dtype = gdf[col].dtype
                        print(f"{i+1:3d}. {col:20s} ({dtype})")
                    break

In [ ]:
#### Configuration ####
# Your node IDs
# reach_ids = usgs_link_merritt.reach_id.drop_duplicates().to_list()
node_ids = nodes.node_id.drop_duplicates().to_list()

# Hydrocron API URL
HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"

# Collection name (Version D data)
COLLECTION_NAME = "SWOT_L2_HR_RiverSP_D"  

# Time range
start_time = "2023-07-27T00:00:00Z"
end_time = "2026-01-31T23:59:59Z"

In [ ]:
# SWOT L2_HR_RiverSP_Node Version D - All 57 fields (2026/01)
all_riversp_node_fields = (
    # Identifiers (5)
    "reach_id,node_id,time,time_tai,time_str,"
    
    # Location (5)
    "lat,lon,lat_u,lon_u,river_name,"
    
    # Water Surface Elevation (6)
    "wse,wse_u,wse_r_u,wse_sm,wse_sm_u,wse_sm_q,wse_sm_q_b,"
    
    # Width (2)
    "width,width_u,"
    
    # Area (5)
    "area_total,area_tot_u,area_detct,area_det_u,area_wse,"
    
    # Geometry/Distance (4)
    "layovr_val,node_dist,xtrk_dist,flow_angle,"
    
    # Quality flags (7)
    "node_q,node_q_b,dark_frac,ice_clim_f,ice_dyn_f,n_good_pix,xovr_cal_q,"
    
    # Radar backscatter (3)
    "rdr_sig0,rdr_sig0_u,rdr_pol,"
    
    # Geophysical corrections (9)
    "geoid_hght,solid_tide,load_tidef,load_tideg,pole_tide,"
    "dry_trop_c,wet_trop_c,iono_c,xovr_cal_c,"
    
    # Prior database info (8)
    "p_wse,p_wse_var,p_width,p_wid_var,p_dist_out,p_length,p_dam_id,p_n_ch_max,p_n_ch_mod,"
    
    # Geometry (1)
    "geometry"
)

all_node_fields = [f.strip() for f in all_riversp_node_fields.split(',') if f.strip()]
print(f"Total Node fields: {len(all_node_fields)}")

In [ ]:
import requests

sample_node = node_ids[0]
HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"
COLLECTION_NAME = "SWOT_L2_HR_RiverSP_D"
start_time = "2025-12-01T00:00:00Z"
end_time = "2026-01-31T23:59:59Z"

valid_fields = []
invalid_fields = []

for field in all_node_fields:
    params = {
        "feature": "Node",
        "feature_id": sample_node,
        "output": "csv",
        "start_time": start_time,
        "end_time": end_time,
        "fields": f"node_id,{field}",  # reach_id는 항상 포함
        "collection_name": COLLECTION_NAME
    }
    
    response = requests.get(HYDROCRON_URL, params=params)
    
    if response.status_code == 200 and "results" in response.json():
        valid_fields.append(field)
        print(f"✓ {field}")
    else:
        invalid_fields.append(field)
        print(f"✗ {field}")

print("\n" + "="*50)
print(f"Valid fields ({len(valid_fields)}):")
print(",".join(valid_fields))
print(f"\nInvalid fields ({len(invalid_fields)}):")
print(",".join(invalid_fields))

Group: reach
Variables: dict_keys(['reach_id', 'time', 'time_str', 'd_x_area', 'd_x_area_u', 'slope', 'slope_u', 'slope_r_u', 'slope2', 'slope2_u', 'slope2_r_u', 'width', 'p_width', 'width_u', 'wse', 'wse_u', 'wse_r_u', 'p_length', 'reach_q', 'reach_q_b', 'dark_frac', 'ice_clim_f', 'ice_dyn_f', 'partial_f', 'n_good_nod', 'obs_frac_n', 'xovr_cal_q', 'xtrk_dist'])

Group: node
Variables: dict_keys(['reach_id', 'node_id', 'time', 'time_str', 'd_x_area', 'd_x_area_u', 'slope', 'slope_u', 'slope_r_u', 'slope2', 'slope2_u', 'slope2_r_u', 'width', 'p_width', 'width_u', 'wse', 'wse_u', 'wse_r_u', 'node_q', 'node_q_b', 'dark_frac', 'ice_clim_f', 'ice_dyn_f', 'n_good_pix', 'xovr_cal_q', 'xtrk_dist'])

In [ ]:
@dask.delayed
def query_hydrocron(query_url, node_id, start_time, end_time, fields, collection_name, empty_df):  # add collection_name
    """Query Hydrocron for node-level time series data."""
    
    params = {
        "feature": "Node",
        "feature_id": node_id,
        "output": "csv",
        "start_time": start_time,
        "end_time": end_time,
        "fields": fields,
        "collection_name": collection_name  # add!!!
    }
    
    results = requests.get(query_url, params=params)
    
    if "results" in results.json().keys():
        results_csv = results.json()["results"]["csv"]
        df = pd.read_csv(StringIO(results_csv))
    else:
        df = empty_df
    
    return df

#### Main Execution ####
if __name__ == "__main__":

    fields = all_node_fields
    
    print("Starting SWOT Data Download")
    print(f"Number of nodes: {len(node_ids)}")
    print(f"Time range: {start_time} to {end_time}")
    print(f"Fields: {fields}")
    print()
    
    # Create Dask client
    print("Creating Dask client...")
    client = Client(n_workers=48)  # Adjust n_workers based on your CPU cores
    print(f"Dask dashboard: {client.dashboard_link}")
    print()
    
    try:
        # Get metadata from first successful query
        print("Getting metadata schema...")
        meta_df = None
        for sample_node in node_ids[:10]:  
            sample_params = {
                "feature": "Node",
                "feature_id": sample_node,
                "output": "csv",
                "start_time": start_time,
                "end_time": end_time,
                "fields": fields,
                "collection_name": COLLECTION_NAME
            }
            
            try:
                sample_result = requests.get(HYDROCRON_URL, params=sample_params, timeout=30)
                if sample_result.status_code == 200 and sample_result.text:
                    sample_json = sample_result.json()
                    if "results" in sample_json.keys() and sample_json["results"]["csv"]:
                        meta_df = pd.read_csv(StringIO(sample_json["results"]["csv"])).head(0)
                        print(f"Got metadata from node {sample_node}")
                        print(f"Columns: {list(meta_df.columns)}")
                        break
            except:
                continue
        
        if meta_df is None:
            raise ValueError("Could not get metadata schema from any node")
        
        # Create queries that return Pandas.DataFrame objects
        print("Creating delayed queries...")
        results = []
        
        for node in node_ids:
            empty_df = meta_df.copy()  
            
            results.append(
                query_hydrocron(
                    HYDROCRON_URL, 
                    node,
                    start_time, 
                    end_time, 
                    all_node_fields, 
                    COLLECTION_NAME,
                    empty_df
                )
            )
        
        print(f"Created {len(results)} queries")
        print()
        
        # Load DataFrame results into dask.dataframe
        print("Loading into Dask DataFrame and computing...")
        ddf = dd.from_delayed(results, meta=meta_df) 
        
        # Compute to get actual data
        df = ddf.compute()
        
        print(f"Downloaded {len(df)} total rows")
        print()
        
        # Remove fill values for missing observations
        print("Cleaning data...")
        df = df.loc[(df["wse"] != -999999999999.0)]
        
        # Convert time_str to datetime format
        df['time_str'] = pd.to_datetime(df['time_str'])
        
        print(f"After cleaning: {len(df)} rows")
        print()
        
        # Save to CSV
        output_file = "swot_data_selected_nodes.parquet"
        df.to_parquet(output_file)#, index=False)
        print(f"Data saved to: {output_file}")
        print()
        
        # Display summary
        print("="*70)
        print("Summary")
        print("="*70)
        print(f"Total observations: {len(df)}")
        print(f"Number of nodes with data: {df['node_id'].nunique()}")
        print()
        print("Observations per node:")
        print(df.groupby('node_id').size().sort_values(ascending=False))
        print()
        
        if 'time_str' in df.columns:
            print(f"Time range:")
            print(f"  Start: {df['time_str'].min()}")
            print(f"  End: {df['time_str'].max()}")
        print()
        
        print("Columns:")
        print(df.columns.tolist())
        print()
        
        print("First few rows:")
        print(df.head(10))
        
    finally:
        # Close client
        print()
        print("Closing Dask client...")
        client.close()
        print("Done!")

In [ ]:
def load_sword_nodes(sword_path):
    """Load and merge all SWORD nodes shapefiles from NA directory."""
    nodes_pattern = str(Path(sword_path) / 'NA' / '*nodes*.shp')
    nodes_files = sorted(glob.glob(nodes_pattern))
    
    if not nodes_files:
        raise FileNotFoundError(f"No nodes shapefiles found in {sword_path}")
    
    # Read and merge all nodes
    nodes_list = [gpd.read_file(f) for f in nodes_files]
    nodes = gpd.GeoDataFrame(
        pd.concat(nodes_list, ignore_index=True),
        crs=nodes_list[0].crs
    )
    
    return nodes

In [ ]:
# Load reaches
sword_path = '/nas/cee-ice/data/SWORD/SWORD_v17b/shp'
nodes = load_sword_nodes(sword_path)

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

target_reaches = df_filtered['reach_id'].unique()
node_ids = nodes[nodes['reach_id'].isin(target_reaches)]['node_id'].to_list()

print(f"Target reaches: {len(target_reaches)}")
print(f"Filtered nodes: {len(node_ids)}")

# Hydrocron API URL
HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"

# Collection name (Version D data)
COLLECTION_NAME = "SWOT_L2_HR_RiverSP_D"  

# Time range
start_time = "2023-07-26T00:00:00Z"
end_time = "2025-12-31T23:59:59Z"

# Fields to download
fields = (
    "reach_id,time_str,node_id,lat,lon,"
    "wse,wse_u,width,width_u,xtrk_dist,flow_angle,crid"
)

In [ ]:
import dask
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
from dask import delayed
import pandas as pd
import numpy as np
import requests
from io import StringIO
import datetime
import time

# Retry decorator for robustness
def query_hydrocron_with_retry(query_url, node_id, start_time, end_time, fields, collection_name, empty_df, max_retries=3):
    """Query Hydrocron with retry logic and timeout."""
    
    params = {
        "feature": "Node",
        "feature_id": node_id,
        "output": "csv",
        "start_time": start_time,
        "end_time": end_time,
        "fields": fields,
        "collection_name": collection_name
    }
    
    for attempt in range(max_retries):
        try:
            results = requests.get(query_url, params=params, timeout=60)  # 60초 타임아웃
            results.raise_for_status()
            
            if "results" in results.json().keys():
                results_csv = results.json()["results"]["csv"]
                df = pd.read_csv(StringIO(results_csv))
                return df
            else:
                return empty_df
                
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
                continue
            else:
                print(f"Failed after {max_retries} attempts for node {node_id}: {e}")
                return empty_df
        except Exception as e:
            print(f"Error for node {node_id}: {e}")
            return empty_df
    
    return empty_df


def process_batch(node_batch, query_url, start_time, end_time, fields, collection_name):
    """Process a batch of nodes sequentially."""
    results = []
    
    for node in node_batch:
        empty_df = pd.DataFrame({
            "node_id": np.int64(node),
            "time_str": datetime.datetime(1900, 1, 1).strftime("%Y-%m-%dT%H:%M:%S"),
            "wse": -999999999999.0,
            "crid": "",
        }, index=[0])
        
        df = query_hydrocron_with_retry(
            query_url, node, start_time, end_time, 
            fields, collection_name, empty_df
        )
        results.append(df)
    
    if results:
        return pd.concat(results, ignore_index=True)
    else:
        return pd.DataFrame()


if __name__ == "__main__":
    
    print("="*70)
    print("Starting SWOT Data Download")
    print("="*70)
    print(f"Time range: {start_time} to {end_time}")
    print(f"Fields: {fields}")
    print(f"Total nodes: {len(node_ids)}")
    print("="*70)
    print()
    
    # Batch nodes - 각 task가 100개 node 처리
    BATCH_SIZE = 100
    node_batches = [node_ids[i:i+BATCH_SIZE] for i in range(0, len(node_ids), BATCH_SIZE)]
    print(f"Created {len(node_batches)} batches of ~{BATCH_SIZE} nodes each")
    
    # Create cluster with reasonable settings
    print("Creating Dask cluster...")
    cluster = LocalCluster(
        n_workers=16,  # 48 -> 16으로 줄임
        threads_per_worker=2,
        memory_limit='4GB',
        silence_logs=False,
    )
    client = Client(cluster)
    print(f"Dask dashboard: {client.dashboard_link}")
    print()
    
    try:
        # Create delayed tasks for batches (not individual nodes!)
        print("Creating delayed batch queries...")
        delayed_results = []
        
        for batch in node_batches:
            delayed_results.append(
                delayed(process_batch)(
                    batch, HYDROCRON_URL, start_time, end_time, 
                    fields, COLLECTION_NAME
                )
            )
        
        print(f"Created {len(delayed_results)} delayed tasks")
        print()
        
        # Compute with progress
        print("Computing...")
        from dask.diagnostics import ProgressBar
        
        with ProgressBar():
            results = dask.compute(*delayed_results)
        
        # Concatenate all results
        df = pd.concat([r for r in results if len(r) > 0], ignore_index=True)
        
        print(f"Downloaded {len(df)} total rows")
        print()
        
        # Clean data
        print("Cleaning data...")
        df = df.loc[(df["wse"] != -999999999999.0)]
        df['time_str'] = pd.to_datetime(df['time_str'])
        
        print(f"After cleaning: {len(df)} rows")
        
        # Save
        output_file = "swot_data_selected_nodes.parquet"
        df.to_parquet(output_file)
        print(f"Data saved to: {output_file}")
        
        # Summary
        print("="*70)
        print("Summary")
        print("="*70)
        print(f"Total observations: {len(df)}")
        print(f"Number of nodes with data: {df['node_id'].nunique()}")
        
    finally:
        print("Closing Dask client...")
        client.close()
        cluster.close()
        print("Done!")

In [ ]:
# # Initial setting
# sample_reach = reach_ids[0]
# sample_params = {
#     "feature": "Reach",
#     "feature_id": sample_reach,
#     "output": "csv",
#     "start_time": start_time,
#     "end_time": end_time,
#     "fields": fields,
#     "collection_name": COLLECTION_NAME
# }

# sample_result = requests.get(HYDROCRON_URL, params=sample_params)
# if "results" in sample_result.json().keys():
#     sample_csv = sample_result.json()["results"]["csv"]
#     meta_df = pd.read_csv(StringIO(sample_csv)).head(0)  # empty df with columns
# else:
#     meta_df = pd.DataFrame(...)

# ddf = dd.from_delayed(results, meta=meta_df)

In [ ]:
# @dask.delayed
# def query_hydrocron(query_url, node_id, start_time, end_time, fields, collection_name, empty_df):  # collection_name 추가
#     """Query Hydrocron for reach-level time series data."""
    
#     params = {
#         "feature": "Node",
#         "feature_id": node_id,
#         "output": "csv",
#         "start_time": start_time,
#         "end_time": end_time,
#         "fields": fields,
#         "collection_name": collection_name  # ← 추가!
#     }
    
#     results = requests.get(query_url, params=params)
    
#     if "results" in results.json().keys():
#         results_csv = results.json()["results"]["csv"]
#         df = pd.read_csv(StringIO(results_csv))
#     else:
#         df = empty_df
    
#     return df

# # =============================================================================
# # Main Execution
# # =============================================================================

# if __name__ == "__main__":
    
#     print("="*70)
#     print("Starting SWOT Data Download")
#     print("="*70)
#     # print(f"Number of reaches: {len(reach_ids)}")
#     print(f"Time range: {start_time} to {end_time}")
#     print(f"Fields: {fields}")
#     print("="*70)
#     print()
    
#     # Create Dask client
#     print("Creating Dask client...")
#     client = Client(n_workers=48)  # Adjust n_workers based on your CPU cores
#     print(f"Dask dashboard: {client.dashboard_link}")
#     print()
    
#     try:
#         # Create queries that return Pandas.DataFrame objects
#         print("Creating delayed queries...")
#         results = []
        
#         for node in node_ids:
#             # Create an empty dataframe for cases where no data is returned
#             empty_df = pd.DataFrame({
#                 "node_id": np.int64(node),
#                 "time_str": datetime.datetime(1900, 1, 1).strftime("%Y-%m-%dT%H:%M:%S"),
#                 "wse": -999999999999.0,
#                 "crid": "",
#                 # "wse_units": "m"
#             }, index=[0])
            
#             results.append(
#                 query_hydrocron(
#                     HYDROCRON_URL, 
#                     node, 
#                     start_time, 
#                     end_time, 
#                     fields, 
#                     COLLECTION_NAME,  # ← 추가!
#                     empty_df
#                 )
#             )
        
#         print(f"Created {len(results)} queries")
#         print()
        
#         # Load DataFrame results into dask.dataframe
#         print("Loading into Dask DataFrame and computing...")
#         ddf = dd.from_delayed(results)
        
#         # Compute to get actual data
#         df = ddf.compute()
        
#         print(f"Downloaded {len(df)} total rows")
#         print()
        
#         # Remove fill values for missing observations
#         print("Cleaning data...")
#         df = df.loc[(df["wse"] != -999999999999.0)]
        
#         # Convert time_str to datetime format
#         df['time_str'] = pd.to_datetime(df['time_str'])
        
#         print(f"After cleaning: {len(df)} rows")
#         print()
        
#         # Save to CSV
#         output_file = "swot_data_selected_nodes.parquet"
#         df.to_parquet(output_file)#, index=False)
#         print(f"Data saved to: {output_file}")
#         print()
        
#         # Display summary
#         print("="*70)
#         print("Summary")
#         print("="*70)
#         print(f"Total observations: {len(df)}")
#         print(f"Number of nodes with data: {df['node_id'].nunique()}")
#         print()
#         print("Observations per node:")
#         print(df.groupby('node_id').size().sort_values(ascending=False))
#         print()
        
#         if 'time_str' in df.columns:
#             print(f"Time range:")
#             print(f"  Start: {df['time_str'].min()}")
#             print(f"  End: {df['time_str'].max()}")
#         print()
        
#         print("Columns:")
#         print(df.columns.tolist())
#         print()
        
#         print("First few rows:")
#         print(df.head(10))
        
#     finally:
#         # Close client
#         print()
#         print("Closing Dask client...")
#         client.close()
#         print("Done!")

In [ ]:
# df_filtered_r = pd.read_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_swot_qc_filtered_vD.parquet')
# reach_time = df_filtered_r[['reach_id','time_str']]
# reach_node = df.merge(reach_time, on =['reach_id','time_str'], how = 'inner')
# reach_node.to_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_swot_qc_filtered_vD_node.parquet')

In [ ]:
df.to_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/selected_swot_vD_nodeonly_ver2.parquet')

In [ ]:
df

In [ ]:
# 날짜만 추출
df['date'] = df['time_str'].dt.date

# 전체 날짜 범위 생성
all_dates = pd.date_range(df['time_str'].min(), df['time_str'].max(), freq='D').date

# 실제 있는 날짜
existing_dates = set(df['date'].unique())

# 빠진 날짜
missing_dates = sorted(set(all_dates) - existing_dates)
print(f"Missing dates: {len(missing_dates)}")

# 빠진 구간 찾기 (연속된 빠진 날짜를 구간으로)
from itertools import groupby
from datetime import timedelta

def find_gaps(missing_dates):
    if not missing_dates:
        return []
    
    gaps = []
    missing_dates = [pd.Timestamp(d) for d in missing_dates]
    
    start = missing_dates[0]
    end = missing_dates[0]
    
    for d in missing_dates[1:]:
        if d - end == timedelta(days=1):
            end = d
        else:
            gaps.append((start, end, (end - start).days + 1))
            start = d
            end = d
    gaps.append((start, end, (end - start).days + 1))
    
    return gaps

gaps = find_gaps(missing_dates)

print("\nMissing date ranges:")
for start, end, days in gaps:
    print(f"  {start.date()} ~ {end.date()} ({days} days)")

### Filtering out (11 filters)

In [ ]:
df_swot = df.copy()

In [ ]:
def apply_harlan_filters(df_swot, waterbody_type='river', hampel_k=2, hampel_gap_days=5, verbose=True):
    """
    Apply 11 quality filters following Harlan et al. (2025)
    
    Parameters:
    -----------
    df_swot : DataFrame
        SWOT data from Hydrocron API
    waterbody_type : str, default='river'
        'river' or 'lake' - affects size threshold
    hampel_k : int, default=2
        Half-window size for Hampel filter
        Window size = 2*k+1
    hampel_gap_days : float, default=5
        Maximum time gap (in days) between consecutive observations
    verbose : bool, default=True
        If False, suppress all print statements
    
    Returns:
    --------
    filtered_df : DataFrame
        Filtered observations
    filter_stats : dict
        Statistics for each filter
    """
    
    hampel_window_size = 2 * hampel_k + 1
    
    if verbose:
        print(f"Original data: {len(df_swot):,} observations")
        print(f"Waterbody type: {waterbody_type}")
        print(f"Hampel filter settings: K={hampel_k} (window={hampel_window_size} points), gap threshold={hampel_gap_days} days")
    
    # Initialize
    df = df_swot.copy()
    filter_stats = {}
    
    # Store Hampel parameters
    filter_stats['hampel_parameters'] = {
        'k': hampel_k,
        'window_size': hampel_window_size,
        'gap_threshold_days': hampel_gap_days,
        'min_sequence_length': hampel_window_size
    }
    
    # ============================================================
    # PHASE 1: Design Limitation Filters
    # ============================================================
    
    if verbose:
        print("\n" + "="*70)
        print("PHASE 1: DESIGN LIMITATION FILTERS")
        print("="*70)
    
    # Filter 1: Ice Climatology Flag < 1
    mask_ice = df['ice_clim_f'] < 1
    n_ice_pass = mask_ice.sum()
    n_ice_fail = (~mask_ice).sum()
    filter_stats['ice_climatology'] = {
        'pass': n_ice_pass,
        'fail': n_ice_fail,
        'pass_rate': n_ice_pass / len(df) * 100
    }
    if verbose:
        print(f"1. Ice Climatology (<1):")
        print(f"   Pass: {n_ice_pass:,} | Fail: {n_ice_fail:,} | Rate: {filter_stats['ice_climatology']['pass_rate']:.1f}%")
    
    # Filter 2: Cross Track Distance 10-60 km
    mask_xtrk = (df['xtrk_dist'].abs() >= 10000) & (df['xtrk_dist'].abs() <= 60000)
    n_xtrk_pass = mask_xtrk.sum()
    n_xtrk_fail = (~mask_xtrk).sum()
    filter_stats['cross_track'] = {
        'pass': n_xtrk_pass,
        'fail': n_xtrk_fail,
        'pass_rate': n_xtrk_pass / len(df) * 100
    }
    if verbose:
        print(f"2. Cross Track Distance (10-60 km):")
        print(f"   Pass: {n_xtrk_pass:,} | Fail: {n_xtrk_fail:,} | Rate: {filter_stats['cross_track']['pass_rate']:.1f}%")
    
    # Filter 3: Waterbody Size Thresholds
    if waterbody_type == 'river':
        mask_size = (df['p_width'] >= 80) & (df['p_length'] >= 7000)
        size_desc = "River (width≥80m, length≥7km)"
    else:
        df['area_km2'] = df['area_total'] / 1e6
        mask_size = df['area_km2'] >= 0.0625
        size_desc = "Lake (area≥0.0625 km²)"
    
    n_size_pass = mask_size.sum()
    n_size_fail = (~mask_size).sum()
    filter_stats['waterbody_size'] = {
        'pass': n_size_pass,
        'fail': n_size_fail,
        'pass_rate': n_size_pass / len(df) * 100
    }
    if verbose:
        print(f"3. Waterbody Size ({size_desc}):")
        print(f"   Pass: {n_size_pass:,} | Fail: {n_size_fail:,} | Rate: {filter_stats['waterbody_size']['pass_rate']:.1f}%")
    
    # Apply all design limitation filters
    mask_design = mask_ice & mask_xtrk & mask_size
    df_design = df[mask_design].copy()
    
    adjusted_denominator = len(df_design)
    n_design_removed = len(df) - adjusted_denominator
    
    if verbose:
        print(f"\n{'─'*70}")
        print(f"✓ Design-filtered dataset: {adjusted_denominator:,} observations")
        print(f"  Removed: {n_design_removed:,} ({n_design_removed/len(df)*100:.1f}%)")
        print(f"  📌 This is the ADJUSTED DENOMINATOR (TRUE 100%)")
        print(f"{'─'*70}")
    
    if adjusted_denominator == 0:
        if verbose:
            print("\n⚠️  WARNING: No observations passed design filters!")
        return df_design, filter_stats
    
    # ============================================================
    # PHASE 2: Quality Filters
    # ============================================================
    
    if verbose:
        print("\n" + "="*70)
        print("PHASE 2: QUALITY FILTERS (on design-filtered data)")
        print("="*70)
    
    df_quality = df_design.copy()
    
    # Filter 4: Summary Quality Flag < 4
    mask_summary = df_quality['reach_q'] < 4
    n_summary_pass = mask_summary.sum()
    filter_stats['summary_quality'] = {
        'pass': n_summary_pass,
        'pass_rate': n_summary_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"4. Summary Quality (<4):")
        print(f"   Pass: {n_summary_pass:,} / {adjusted_denominator:,} ({filter_stats['summary_quality']['pass_rate']:.1f}%)")
    
    # Filter 5: Partial Coverage Flag = 0
    mask_partial = df_quality['partial_f'] == 0
    n_partial_pass = mask_partial.sum()
    filter_stats['partial_coverage'] = {
        'pass': n_partial_pass,
        'pass_rate': n_partial_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"5. Partial Coverage (=0):")
        print(f"   Pass: {n_partial_pass:,} / {adjusted_denominator:,} ({filter_stats['partial_coverage']['pass_rate']:.1f}%)")
    
    # Filter 6: Crossover Calibration Quality < 2
    if 'xovr_cal_q' in df_quality.columns:
        mask_xovr = df_quality['xovr_cal_q'] < 2
        n_xovr_pass = mask_xovr.sum()
        filter_stats['crossover_cal'] = {
            'pass': n_xovr_pass,
            'pass_rate': n_xovr_pass / adjusted_denominator * 100
        }
        if verbose:
            print(f"6. Crossover Calibration (<2):")
            print(f"   Pass: {n_xovr_pass:,} / {adjusted_denominator:,} ({filter_stats['crossover_cal']['pass_rate']:.1f}%)")
    else:
        mask_xovr = pd.Series([True] * len(df_quality), index=df_quality.index)
        filter_stats['crossover_cal'] = {'pass': len(df_quality), 'pass_rate': 100.0}
        if verbose:
            print("6. Crossover Calibration: ⚠️  Field not available (skipping)")
    
    # Filter 7: Bitwise Quality Flag < 4,194,304
    mask_bitwise = df_quality['reach_q_b'] < 4194304
    n_bitwise_pass = mask_bitwise.sum()
    filter_stats['bitwise_quality'] = {
        'pass': n_bitwise_pass,
        'pass_rate': n_bitwise_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"7. Bitwise Quality (<4,194,304):")
        print(f"   Pass: {n_bitwise_pass:,} / {adjusted_denominator:,} ({filter_stats['bitwise_quality']['pass_rate']:.1f}%)")
    
    # Filter 8: Dark Water Fraction ≤ 0.5
    mask_dark = df_quality['dark_frac'] <= 0.5
    n_dark_pass = mask_dark.sum()
    filter_stats['dark_water'] = {
        'pass': n_dark_pass,
        'pass_rate': n_dark_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"8. Dark Water Fraction (≤0.5):")
        print(f"   Pass: {n_dark_pass:,} / {adjusted_denominator:,} ({filter_stats['dark_water']['pass_rate']:.1f}%)")
    
    # Filter 9: Observed Area Fraction >= 0.5
    df_quality['obs_area_frac'] = df_quality['area_detct'] / df_quality['area_total']
    mask_area = df_quality['obs_area_frac'] >= 0.5
    n_area_pass = mask_area.sum()
    filter_stats['observed_area'] = {
        'pass': n_area_pass,
        'pass_rate': n_area_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"9. Observed Area Fraction (>=0.5):")
        print(f"   Pass: {n_area_pass:,} / {adjusted_denominator:,} ({filter_stats['observed_area']['pass_rate']:.1f}%)")
    
    # Filter 10: WSE Prior Difference < 50m
    df_quality['wse_prior_diff'] = (df_quality['wse'] - df_quality['p_wse']).abs()
    mask_prior = df_quality['wse_prior_diff'] < 50
    n_prior_pass = mask_prior.sum()
    filter_stats['wse_prior'] = {
        'pass': n_prior_pass,
        'pass_rate': n_prior_pass / adjusted_denominator * 100
    }
    if verbose:
        print(f"10. WSE Prior Difference (<50m):")
        print(f"   Pass: {n_prior_pass:,} / {adjusted_denominator:,} ({filter_stats['wse_prior']['pass_rate']:.1f}%)")
    
    # Combine quality filters (excluding Hampel)
    mask_quality_pre_hampel = (
        mask_summary & mask_partial & mask_xovr & mask_bitwise & 
        mask_dark & mask_area & mask_prior
    )
    df_pre_hampel = df_quality[mask_quality_pre_hampel].copy()
    n_pre_hampel = len(df_pre_hampel)
    
    if verbose:
        print(f"\n{'─'*70}")
        print(f"After filters 4-10: {n_pre_hampel:,} / {adjusted_denominator:,} ({n_pre_hampel/adjusted_denominator*100:.1f}%)")
        print(f"{'─'*70}")
    
    # ============================================================
    # Filter 11: Hampel Outlier Filter
    # ============================================================
    
    if verbose:
        print(f"\n11. Hampel Outlier Filter:")
        print(f"   Settings: h=3, K={hampel_k}, window={hampel_window_size} points, gap threshold={hampel_gap_days} days")
        print(f"   🔧 Calculating per-reach outliers with temporal gap detection...")
    
    df_pre_hampel['hampel_outlier'] = 0
    hampel_stats = {
        'total_checked': 0, 
        'outliers_found': 0, 
        'reaches_processed': 0,
        'sequences_found': 0,
        'sequences_too_short': 0
    }
    
    # Apply Hampel filter per reach
    for reach_id in df_pre_hampel['reach_id'].unique():
        reach_mask = df_pre_hampel['reach_id'] == reach_id
        reach_data = df_pre_hampel[reach_mask].sort_values('time_str').copy()
        
        if len(reach_data) < hampel_window_size:
            continue
        
        hampel_stats['reaches_processed'] += 1
        sequences = split_into_temporal_sequences(reach_data, gap_threshold_days=hampel_gap_days)
        hampel_stats['sequences_found'] += len(sequences)
        
        for seq_indices in sequences:  # ← 들여쓰기 수정
            seq_data = reach_data.loc[seq_indices]
            wse_series = seq_data['wse'].values
            
            outlier_indices = hampel_filter_harlan(wse_series, h=3, k=hampel_k)
            
            if len(outlier_indices) > 0:
                original_indices = seq_indices[outlier_indices]
                df_pre_hampel.loc[original_indices, 'hampel_outlier'] = 1
                hampel_stats['outliers_found'] += len(outlier_indices)
            
            hampel_stats['total_checked'] += len(seq_indices)

    mask_hampel = df_pre_hampel['hampel_outlier'] == 0
    n_hampel_pass = mask_hampel.sum()
    n_hampel_fail = (~mask_hampel).sum()

    filter_stats['hampel_outlier'] = {
        'pass': n_hampel_pass,
        'fail': n_hampel_fail,
        'pass_rate': n_hampel_pass / adjusted_denominator * 100,
        'reaches_processed': hampel_stats['reaches_processed'],
        'sequences_found': hampel_stats['sequences_found'],
        'sequences_too_short': hampel_stats['sequences_too_short']
    }
    
    if verbose:
        print(f"   Reaches processed: {hampel_stats['reaches_processed']}")
        print(f"   Temporal sequences found: {hampel_stats['sequences_found']}")
        print(f"   Sequences too short (<{hampel_window_size} obs): {hampel_stats['sequences_too_short']}")
        print(f"   Outliers detected: {n_hampel_fail:,} / {hampel_stats['total_checked']:,} ({n_hampel_fail/max(1,hampel_stats['total_checked'])*100:.1f}%)")
        print(f"   Pass: {n_hampel_pass:,} / {adjusted_denominator:,} ({filter_stats['hampel_outlier']['pass_rate']:.1f}%)")
        print(f"   💡 Note: {hampel_window_size}-point window (K={hampel_k}), sequences split at >{hampel_gap_days} day gaps")
    
    # Final filtered dataset
    filtered_df = df_pre_hampel[mask_hampel].copy()
    final_count = len(filtered_df)
    
    # Summary statistics
    if verbose:
        print(f"\n{'='*70}")
        print(f"FILTERING SUMMARY")
        print(f"{'='*70}")
        print(f"Original observations:              {len(df):>10,}")
        print(f"Design limitation filters removed:  {n_design_removed:>10,} ({n_design_removed/len(df)*100:>5.1f}%)")
        print(f"Adjusted denominator (TRUE 100%):   {adjusted_denominator:>10,}")
        print(f"Quality filters removed:            {adjusted_denominator - final_count:>10,} ({(adjusted_denominator - final_count)/adjusted_denominator*100:>5.1f}%)")
        print(f"Final observations:                 {final_count:>10,}")
        print(f"\nOverall pass-through rate:")
        print(f"  - vs. adjusted denominator:       {final_count/adjusted_denominator*100:>10.1f}%")
        print(f"  - vs. original data:              {final_count/len(df)*100:>10.1f}%")
        print(f"\nHampel filter configuration:")
        print(f"  - K={hampel_k}, Window={hampel_window_size} points, Gap threshold={hampel_gap_days} days")
        print(f"{'='*70}")
    
    filter_stats['summary'] = {
        'original_count': len(df),
        'design_removed': n_design_removed,
        'adjusted_denominator': adjusted_denominator,
        'quality_removed': adjusted_denominator - final_count,
        'final_count': final_count,
        'pass_rate_adjusted': final_count / adjusted_denominator * 100 if adjusted_denominator > 0 else 0,
        'pass_rate_original': final_count / len(df) * 100
    }
    
    return filtered_df, filter_stats


def split_into_temporal_sequences(reach_data, gap_threshold_days=5):
    """Split reach data into continuous temporal sequences based on time gaps"""
    import pandas as pd
    
    if not pd.api.types.is_datetime64_any_dtype(reach_data['time_str']):
        reach_data = reach_data.copy()
        reach_data['time_str'] = pd.to_datetime(reach_data['time_str'])
    
    times = reach_data['time_str'].values
    indices = reach_data.index.values
    
    sequences = []
    current_sequence = [indices[0]]
    
    for i in range(1, len(indices)):
        time_diff = (times[i] - times[i-1]) / np.timedelta64(1, 'D')
        
        if time_diff <= gap_threshold_days:
            current_sequence.append(indices[i])
        else:
            sequences.append(np.array(current_sequence))
            current_sequence = [indices[i]]
    
    if len(current_sequence) > 0:
        sequences.append(np.array(current_sequence))
    
    return sequences


def hampel_filter_harlan(series, h=3, k=2):
    """Hampel outlier filter with one-sided windows at edges"""
    import numpy as np
    
    window_size = 2 * k + 1
    
    # if len(series) < window_size:
    #     return []
    
    series = np.array(series)
    n = len(series)
    outliers = []
    
    # Process ALL points including edges
    for i in range(n):
        # Determine window boundaries
        if i < k:
            # Left edge: extend window to right
            window_start = 0
            window_end = min(window_size, n)
        elif i >= n - k:
            # Right edge: extend window to left
            window_start = max(0, n - window_size)
            window_end = n
        else:
            # Center: symmetric window
            window_start = i - k
            window_end = i + k + 1
        
        window = series[window_start:window_end]
        median_i = np.median(window)
        mad_i = np.median(np.abs(window - median_i))
        
        if mad_i == 0:
            if series[i] != median_i:
                outliers.append(i)
            continue
        
        scaled_mad = 1.4826 * mad_i
        deviation = abs(series[i] - median_i)
        
        if deviation > h * scaled_mad:
            outliers.append(i)
    
    return outliers


def test_hampel_settings(df, k_values=[1, 2, 3, 5, 10], gap_days=[3, 5, 7, 10]):
    """
    Test multiple combinations of K and gap_days - ONLY PRINTS COMPARISON TABLE
    
    Parameters:
    -----------
    df : DataFrame
        SWOT data
    k_values : list of int
        K values to test
    gap_days : list of float
        Gap threshold values to test (in days)
    """
    results = []
    
    for k in k_values:
        for gap in gap_days:
            # Run filtering with verbose=False (no printing)
            df_filtered, stats = apply_harlan_filters(
                df, 
                waterbody_type='river',
                hampel_k=k,
                hampel_gap_days=gap,
                verbose=False  # ← SUPPRESS ALL OUTPUT
            )
            
            results.append({
                'k': k,
                'window_size': 2*k+1,
                'gap_days': gap,
                'final_count': stats['summary']['final_count'],
                'pass_rate': stats['summary']['pass_rate_adjusted'],
                'outliers': stats['hampel_outlier']['fail'],
                'sequences': stats['hampel_outlier']['sequences_found']
            })
    
    # ONLY print comparison table
    print("="*84)
    print("COMPARISON SUMMARY")
    print("="*84)
    print(f"{'K':<4} {'Window':<8} {'Gap(d)':<8} {'Final':<10} {'Pass%':<8} {'Outliers':<10} {'Sequences':<10}")
    print("─"*84)
    
    for r in results:
        print(f"{r['k']:<4} {r['window_size']:<8} {r['gap_days']:<8} "
              f"{r['final_count']:<10,} {r['pass_rate']:<8.1f} "
              f"{r['outliers']:<10,} {r['sequences']:<10}")
    
    return pd.DataFrame(results)


# ============================================================
# USAGE
# ============================================================

# Only comparison table will be printed
comparison_df = test_hampel_settings(
    df_swot, 
    k_values=[1, 2, 5, 7], 
    gap_days=[1, 3, 5, 9999]
)

In [ ]:
df_filtered, stats = apply_harlan_filters(
    df_swot, 
    hampel_k=7, 
    hampel_gap_days=3
)

df_filtered.to_parquet('/work/pi_kandread_umass_edu/swot-urban/2_urban_reaches/swotall_qc_filtered_vD_nodeonly_ver2.parquet')